In [ ]:
import re
import numpy as np
from gensim.models import KeyedVectors

# 1. 파일 경로 설정
tsv_path = 'data/ko.tsv'
output_path = 'data/ko.kv'

words = []
vectors = []

# 2. tsv 파일에서 단어와 벡터를 추출하는 정규식 패턴
# 형식 예시: ID \t Word \t [ v1 v2 ... ]
word_start_pattern = re.compile(r'^(\d+)\t([^\t]+)\t\[(.+)$')

current_word = None
current_vec = []

print("데이터 분석 및 추출 시작...")

with open(tsv_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
            
        m = word_start_pattern.match(line)
        if m:
            # 새로운 단어가 나오면 이전 단어의 벡터 저장
            if current_word:
                words.append(current_word)
                vectors.append(np.array(current_vec, dtype=np.float32))
            
            # 새 단어 정보 저장 시작
            current_word = m.group(2)
            segment = m.group(3).rstrip(']')
            current_vec = [float(x) for x in re.findall(r'-?\d+\.\d+(?:e[+-]\d+)?', segment)]
        else:
            # 벡터 값이 여러 줄에 걸쳐 있을 경우 계속 추가
            segment = line.rstrip(']')
            current_vec.extend([float(x) for x in re.findall(r'-?\d+\.\d+(?:e[+-]\d+)?', segment)])

# 마지막 단어 처리
if current_word:
    words.append(current_word)
    vectors.append(np.array(current_vec, dtype=np.float32))

# 3. Gensim의 KeyedVectors 객체로 재조립
dim = len(vectors[0]) # 200차원 확인
kv = KeyedVectors(dim)
kv.add_vectors(words, vectors)

# 4. 현대식 로드 파일(.kv)로 저장
kv.save(output_path)

print(f"✅ 변환 완료! 총 {len(words)}개의 단어 벡터가 {output_path}에 저장되었습니다.")
